In [1]:
from pathlib import Path
from datetime import datetime
from circumplex import octants
import xarray as xr
from pipefunc import pipefunc, PipeFunc, Pipeline
import pandas as pd

from scm_inst import scm as scm_inst

today = datetime.now().strftime("%Y-%m-%d")

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR.joinpath("data")
OUTPUT_DIR = PROJECT_DIR.joinpath("output/" + today)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCALES = ["PAQ1", "PAQ2", "PAQ3", "PAQ4", "PAQ5", "PAQ6", "PAQ7", "PAQ8"]
EQ_ANGLES = octants()

satp = pd.read_excel(
    DATA_DIR.joinpath("SATP Dataset v1.5.xlsx"), sheet_name="Main Merge"
)
satp.drop(columns=["Gender", "Age", "sequence_id", "loud"], inplace=True)


In [2]:
index_cols = ["Language", "Institution", "Participant", "Recording"]

multiindex = pd.MultiIndex.from_frame(satp[index_cols])

satp.loc[multiindex.duplicated(), "Participant"] = "FER_5M04a"
multiindex = pd.MultiIndex.from_frame(satp[index_cols])

In [3]:
variables = SCALES
satp_multi = satp.set_index(multiindex)
satp_multi.drop(columns=index_cols, inplace=True)
satp_multi = satp_multi.dropna(subset=SCALES)

satp_multi

PAQ1  PAQ2  PAQ3  PAQ4   PAQ5  \
Language Institution Participant Recording                                  
arb      BIS         BIS_1       CG01       57.0  23.0  68.0   7.0   17.0   
                                 CG04       84.0  76.0  84.0  43.0   13.0   
                                 CG07       14.0  15.0  74.0  66.0   56.0   
                                 CT301       5.0  14.0  14.0  94.0   85.0   
                                 E01b       14.0  15.0  23.0  82.0   83.0   
...                                          ...   ...   ...   ...    ...   
zsm      UPM         UPM_59      KT01       47.0  33.0  31.0  48.0   58.0   
                                 E10        42.0  30.0  83.0  66.0   58.0   
                                 W01        15.0  86.0  70.0  92.0   87.0   
                                 E11b        8.0  87.0  98.0  99.0  100.0   
                                 HR01        9.0  85.0  92.0  96.0  100.0   

                                            PAQ6  PAQ7  PAQ8  
Language Institution Participant Recording                    
arb      BIS         BIS_1       CG01        7.0  16.0  75.0  
                                 CG04       15.0  25.0  45.0  
                                 CG07       73.0  23.0  15.0  
                                 CT301      94.0  75.0   4.0  
                                 E01b       84.0  76.0  64.0  
...                                          ...   ...   ...  
zsm      UPM         UPM_59      KT01       83.0  71.0  43.0  
                                 E10        41.0  19.0  34.0  
                                 W01        44.0  35.0   4.0  
                                 E11b       16.0   5.0   0.0  
                                 HR01       45.0  13.0   0.0  

[19065 rows x 8 columns]

## Ipsatization

In [11]:
satp_multi.mean()

PAQ1    45.271922
PAQ2    45.664881
PAQ3    50.685840
PAQ4    44.541594
PAQ5    44.521277
PAQ6    46.069058
PAQ7    41.247177
PAQ8    41.225834
dtype: float64

In [4]:
parts_mean = satp.groupby("Participant")[SCALES].mean()

In [ ]:
satp[SCALES].